# Experiment 8: Speech Recognition and Text-to-Speech Pipeline Using Whisper and ElevenLabs

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To build an end-to-end voice pipeline that transcribes audio using OpenAI Whisper (ASR), performs LLM-based processing on the transcript, and synthesises a spoken response using text-to-speech (TTS), creating a voice assistant prototype.

## 2. Objectives

- To understand the three stages of a voice assistant: speech to text, language model, text to speech.
- To transcribe speech with Whisper models of different sizes.
- To measure transcription quality with Word Error Rate (WER) and speed with the real-time factor.
- To study the effect of background noise on recognition.
- To generate a reply with an LLM and convert it to speech.
- To connect the stages into one working pipeline.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- Internet connection (model download, Gemini API and the gTTS speech service)
- API keys: Gemini (language model); ElevenLabs (optional, for the neural voice)
- Required Python libraries: `faster-whisper`, `jiwer`, `soundfile`, `librosa`, `openai`, `gTTS`, `elevenlabs`, `pandas`

## 4. Theory

### A. The voice assistant pipeline

`microphone audio -> ASR (Whisper) -> text -> LLM -> reply text -> TTS -> speaker audio`

Each stage can be replaced independently, and each adds delay, so the total response time is the sum of the three.

### B. Whisper

Whisper is a speech-recognition model trained on 680,000 hours of multilingual audio. It converts 16 kHz audio into text. It comes in sizes (tiny, base, small, medium, large): bigger models make fewer errors but are slower. This experiment uses the `faster-whisper` implementation, which runs the same Whisper models efficiently on a CPU.

### C. Evaluation of speech recognition

| Metric | Definition |
|---|---|
| Word Error Rate (WER) | (substitutions + deletions + insertions) divided by the number of words in the reference transcript. Lower is better; 0 is perfect. |
| Real-time factor (RTF) | Processing time divided by audio duration. Below 1 means faster than real time, which a live assistant needs. |
| Signal-to-noise ratio (SNR) | Loudness of speech relative to noise, in decibels. Lower SNR means a noisier recording. |

### D. Text-to-speech

TTS converts text to a waveform. Classic engines are fast but sound robotic. Neural services such as ElevenLabs produce natural voices and are accessed through an API. This experiment uses Google's gTTS (free, no key) and shows the ElevenLabs call.

## 5. Algorithm / Procedure

1. Install and import the libraries.
2. Load a set of speech recordings with known transcripts (LibriSpeech).
3. Transcribe them with Whisper tiny and base; compute WER and real-time factor.
4. Add noise at several levels and repeat the measurement.
5. Send a transcript to the LLM and obtain a short spoken-style reply.
6. Convert the reply to speech with TTS.
7. Join all stages into one function and run it on spoken questions.
8. Measure the time of every stage and record observations.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q faster-whisper jiwer soundfile librosa openai gTTS elevenlabs pandas

### Step 2: Load the recordings

LibriSpeech has read English speech with exact transcripts. We use 20 clean recordings and 10 harder "other" recordings (more accents and noisier conditions). Whisper expects mono audio at 16 kHz, so every file is resampled.

In [2]:
import os, json, time, string, warnings, numpy as np, pandas as pd, soundfile as sf, librosa, jiwer
from dotenv import load_dotenv, find_dotenv
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 180)
load_dotenv(find_dotenv(usecwd=True))

clips = pd.DataFrame(json.load(open("data/asr_set.json")))

def load_audio(path):
    audio, sr = sf.read(path, dtype="float32")
    audio = audio.mean(axis=1) if audio.ndim > 1 else audio
    return librosa.resample(audio, orig_sr=sr, target_sr=16000) if sr != 16000 else audio

clips["audio"] = clips.file.apply(load_audio)
print(len(clips), "recordings |", round(clips.seconds.sum()), "seconds of audio in total")
clips[["subset", "seconds", "text"]].head(3)

30 recordings | 198 seconds of audio in total


,subset,seconds,text
0,clean,3.5,concord returned to its place amidst the tents
1,clean,3.0,i can perceive love clearly enough
2,clean,14.4,bragelonne watched for some time the conduct of the two lovers lis...


### Step 3: Transcribe with Whisper and measure WER

`WhisperModel(size)` downloads the model. `transcribe` returns segments of text. The word error rate is computed after lower-casing and removing punctuation.

In [3]:
from faster_whisper import WhisperModel

def normalize(s):
    return " ".join(s.lower().translate(str.maketrans("", "", string.punctuation)).split())

def transcribe(model, audio, **kw):
    segments, _ = model.transcribe(audio, **kw)
    return " ".join(s.text.strip() for s in segments).strip()

def evaluate(model, audios, refs):
    t0 = time.time()
    hyps = [transcribe(model, a, language="en") for a in audios]
    seconds = time.time() - t0
    wer = jiwer.wer([normalize(r) for r in refs], [normalize(h) or "<empty>" for h in hyps])
    return wer, seconds / (sum(len(a) for a in audios) / 16000), hyps

models = {size: WhisperModel(size, device="cpu", compute_type="int8") for size in ["tiny", "base"]}
rows = {}
for size, m in models.items():
    wer, rtf, hyps = evaluate(m, list(clips.audio), list(clips.text))
    rows[size] = {"WER": round(wer, 3), "Real-time factor": round(rtf, 2)}
    clips["hyp_" + size] = hyps
size_table = pd.DataFrame(rows).T
size_table

,WER,Real-time factor
tiny,0.071,0.11
base,0.046,0.22


### Step 4: Clean against harder recordings

In [4]:
rows = []
for subset in ["clean", "other"]:
    part = clips[clips.subset == subset]
    for size in models:
        rows.append({"recordings": subset, "model": size, "WER": round(jiwer.wer(list(part.text.map(normalize)), list(part["hyp_" + size].map(normalize))), 3)})
pd.DataFrame(rows).pivot(index="recordings", columns="model", values="WER")

model,base,tiny
recordings,,
clean,0.039,0.072
other,0.065,0.071


### Step 5: Effect of background noise

White noise is added at a chosen signal-to-noise ratio: 20 dB is a quiet room, 5 dB is loud. The `base` model transcribes the noisy clips.

In [5]:
def add_noise(audio, snr_db, seed=0):
    noise_power = np.mean(audio ** 2) / (10 ** (snr_db / 10))
    return (audio + np.random.default_rng(seed).normal(0, np.sqrt(noise_power), audio.shape)).astype("float32")

clean = clips[clips.subset == "clean"]
rows = [{"SNR (dB)": "no noise", "WER": round(jiwer.wer(list(clean.text.map(normalize)), list(clean.hyp_base.map(normalize))), 3)}]
for snr in [20, 10, 5, 0]:
    wer, _, _ = evaluate(models["base"], [add_noise(a, snr) for a in clean.audio], list(clean.text))
    rows.append({"SNR (dB)": snr, "WER": round(wer, 3)})
noise_table = pd.DataFrame(rows).set_index("SNR (dB)")
noise_table

,WER
SNR (dB),
no noise,0.039
20,0.044
10,0.069
5,0.141
0,0.303


### Step 6: The language model stage

The transcript is sent to the LLM with a system prompt that asks for a short answer suited to speech: no lists, no formatting, one or two sentences.

In [6]:
from openai import OpenAI
client = OpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
SYSTEM = "You are a voice assistant. Answer in at most two short spoken sentences. No lists, no markdown, no emojis."

def llm_reply(text):
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model="gemini-2.5-flash-lite", temperature=0, max_tokens=120,
                                               messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}])
            return r.choices[0].message.content.strip()
        except Exception:
            time.sleep(2 ** attempt)
    return ""

print(llm_reply("What is the difference between weather and climate?"))

Weather describes the atmospheric conditions at a specific time and place. Climate, on the other hand, refers to the average weather patterns over a long period in a particular region.


### Step 7: The text-to-speech stage

gTTS converts text to an MP3 file. The same function shape works for ElevenLabs, which needs an API key and gives a more natural neural voice.

In [7]:
from gtts import gTTS

def speak(text, path, tld="co.in"):
    gTTS(text, lang="en", tld=tld).save(path)      # tld "co.in" selects an Indian-English voice
    return path

os.makedirs("out", exist_ok=True)
speak("Weather is what the atmosphere does today, while climate is its average over many years.", "out/reply_gtts.mp3")
print("Saved out/reply_gtts.mp3,", round(os.path.getsize("out/reply_gtts.mp3") / 1024), "KB")

Saved out/reply_gtts.mp3, 51 KB


In [8]:
# Optional: a neural voice from ElevenLabs (needs ELEVENLABS_API_KEY in .env)
from elevenlabs.client import ElevenLabs

if os.environ.get("ELEVENLABS_API_KEY"):
    eleven = ElevenLabs(api_key=os.environ["ELEVENLABS_API_KEY"])
    audio = eleven.text_to_speech.convert(text="Weather is what the atmosphere does today, while climate is its average over many years.",
                                          voice_id="JBFqnCBsd6RMkjVDRZzb", model_id="eleven_flash_v2_5", output_format="mp3_44100_128")
    with open("out/reply_elevenlabs.mp3", "wb") as f:
        for chunk in audio:
            f.write(chunk)
    print("Saved out/reply_elevenlabs.mp3,", round(os.path.getsize("out/reply_elevenlabs.mp3") / 1024), "KB")

Saved out/reply_elevenlabs.mp3, 44 KB


### Step 8: The complete voice assistant

`voice_assistant` takes an audio array and returns the transcript, the reply, the path of the spoken reply and the time of each stage. To test it without a microphone, we first create spoken questions with gTTS (a different accent from the reply voice) and feed them to the pipeline as if a user had spoken them.

In [9]:
def voice_assistant(audio, out_path):
    t0 = time.time(); heard = transcribe(models["base"], audio, language="en")
    t1 = time.time(); reply = llm_reply(heard)
    t2 = time.time(); speak(reply, out_path)
    t3 = time.time()
    return {"heard": heard, "reply": reply, "asr_s": t1 - t0, "llm_s": t2 - t1, "tts_s": t3 - t2}

questions = ["What is the capital of India?", "How many days are there in a leap year?",
             "Explain what a neural network is in simple words.", "Who wrote the play Romeo and Juliet?"]
results = []
for i, q in enumerate(questions):
    gTTS(q, lang="en", tld="com").save(f"out/q{i}.mp3")
    audio, sr = librosa.load(f"out/q{i}.mp3", sr=16000)            # the 'microphone recording'
    results.append({"question": q, **voice_assistant(audio, f"out/a{i}.mp3")})
results = pd.DataFrame(results)
results[["question", "heard", "reply"]]

,question,heard,reply
0,What is the capital of India?,Which is the capital of India?,The capital of India is New Delhi. It is located in the northern p...
1,How many days are there in a leap year?,How many days are there in a leap here?,There are 366 days in a leap year. This extra day is added to Febr...
2,Explain what a neural network is in simple words.,Explain what a neural network is in simple words.,A neural network is a computer system inspired by the human brain ...
3,Who wrote the play Romeo and Juliet?,Who wrote the play Romeo and Juliet?,William Shakespeare wrote the play Romeo and Juliet. It is one of ...


## 7. Experimental Observations

### Observation Table 1: Whisper model size, accuracy and speed (30 recordings)

In [10]:
size_table

,WER,Real-time factor
tiny,0.071,0.11
base,0.046,0.22


### Observation Table 2: Effect of noise on the base model (20 clean recordings)

In [11]:
noise_table

,WER
SNR (dB),
no noise,0.039
20,0.044
10,0.069
5,0.141
0,0.303


### Observation Table 3: Time of each pipeline stage

In [12]:
timing = results[["question", "asr_s", "llm_s", "tts_s"]].copy()
timing["total_s"] = timing[["asr_s", "llm_s", "tts_s"]].sum(axis=1)
timing.assign(question=timing.question.str[:40]).set_index("question").round(2)

,asr_s,llm_s,tts_s,total_s
question,,,,
What is the capital of India?,0.91,1.03,0.30,2.25
How many days are there in a leap year?,1.02,0.97,0.27,2.26
Explain what a neural network is in simp,0.95,0.89,1.23,3.06
Who wrote the play Romeo and Juliet?,0.97,1.04,0.28,2.29


**Observations**

- Whisper `base` is more accurate than `tiny`: WER 0.046 against 0.071 over the 30 recordings (about 5 and 7 wrong words in every 100). The price is speed: the real-time factor is 0.22 for `base` and 0.11 for `tiny`. Both are far below 1, so both are faster than real time on a CPU.
- The harder "other" recordings cost `base` more accuracy (WER 0.065, against 0.039 on the clean ones), while `tiny` is similar on both (0.071 and 0.072), so the larger model gains most when the audio is clean.
- Noise degrades recognition steadily: the WER of `base` rose from 0.039 without noise to 0.044 at 20 dB, 0.069 at 10 dB, 0.141 at 5 dB and 0.303 at 0 dB, where about one word in three is wrong. Up to 20 dB the model is almost unaffected.
- The assistant worked end to end. For the spoken question "How many days are there in a leap year?" Whisper heard "leap here", but the language model still answered correctly (366 days), because the surrounding words gave enough context. The other three questions were transcribed exactly.
- Each spoken exchange took about 2.3 to 3.1 seconds in total: roughly 1.0 s for recognition, 1.0 s for the language-model reply and 0.3 to 1.2 s for speech synthesis (longer replies take longer to synthesise). A conversation feels natural at about this latency, and streaming each stage would reduce the wait further.

## 8. Result

A voice assistant pipeline was built from Whisper speech recognition, a Gemini language model and text-to-speech. Whisper base reached a word error rate of 0.046 on 30 LibriSpeech recordings (tiny: 0.071) at a real-time factor of 0.22, and the error rate of base rose from 0.039 without noise to 0.303 at 0 dB signal-to-noise ratio. The complete assistant transcribed four spoken questions, answered them and produced a spoken reply, with a total response time of about 2.3 to 3.1 seconds per question.

## 9. Conclusion

A voice assistant is a chain of three independent models: Whisper converts speech to text, an LLM produces the answer, and a TTS engine speaks it. Larger Whisper models are more accurate but slower, and recognition quality drops as background noise increases, so the choice of model and the audio quality must be matched to the application. The response time of the whole assistant is the sum of the three stages, and the language-model call and the speech synthesis are usually the largest parts when the speech model runs on suitable hardware.